# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [13]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('/kaggle/input/datasets/advaithkatkojwala/world-population/world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** **Line chart, indexed to 1970 = 100.** Growth *relative to 1970 size* is a change-over-time question, so a line chart with every continent rebased to the same starting point is the fair comparison (raw totals would just show Asia's size). Africa is the one accent colour and the thickest line; the other five continents are grey context. Continents are labelled directly at the line ends, so there is no legend.

In [14]:
# Q1
import plotly.express as px

GREY, ACCENT = '#B8B8B8', '#0072B2'

cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100      # 1970 = 100

fig = px.line(
    cont, x='Year', y='Index', color='Continent', markers=True, template='plotly_white',
    color_discrete_map={c: (ACCENT if c == 'Africa' else GREY) for c in cont['Continent'].unique()},
    title='Africa has nearly quadrupled since 1970 — Europe has grown by only 13%'
          '<br><sup>Population by continent, indexed to 1970 = 100</sup>',
)
fig.for_each_trace(lambda t: t.update(line_width=4 if t.name == 'Africa' else 2,
                                      marker_size=7 if t.name == 'Africa' else 4))
fig.add_hline(y=100, line_dash='dot', line_color='#888')

# direct labels at the right end, nudged apart so they do not overlap
ends = cont[cont['Year'] == 2022].sort_values('Index').reset_index(drop=True)
last = -1e9
for _, r in ends.iterrows():
    y = max(r['Index'], last + 18)
    last = y
    fig.add_annotation(x=2022, y=y, text=f"<b>{r['Continent']}</b> {r['Index']:.0f}", showarrow=False,
                       xanchor='left', xshift=10, yanchor='middle',
                       font=dict(color=ACCENT if r['Continent'] == 'Africa' else '#666'))

fig.update_layout(showlegend=False, margin=dict(r=150), height=500)
fig.update_xaxes(title=None, showgrid=False, tickvals=[1970, 1980, 1990, 2000, 2010, 2020])
fig.update_yaxes(title=None, gridcolor='#EEE')
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** **Sorted horizontal bar chart.** This is a ranking of 15 named countries, and bar length is the most accurate channel for comparing amounts; long country names fit on the y-axis when the bars are horizontal. Bars are sorted so the largest is on top, African countries use the accent blue and all others are grey, so the answer to "how many are in Africa?" can be counted at a glance. Values are labelled on the bars, so the x-axis is hidden.

In [15]:
# Q2
import plotly.express as px

GREY, ACCENT = '#B8B8B8', '#0072B2'

df2 = df.assign(Added=(df['2022 Population'] - df['1970 Population']) / 1e6)      # millions
top = df2.nlargest(15, 'Added').sort_values('Added')                             # ascending -> largest on top
top['Region'] = np.where(top['Continent'] == 'Africa', 'Africa', 'Other')
n_africa = int((top['Continent'] == 'Africa').sum())

fig = px.bar(
    top, x='Added', y='Country', orientation='h', color='Region', text='Added', template='plotly_white',
    color_discrete_map={'Africa': ACCENT, 'Other': GREY},
    title=f'{n_africa} of the 15 countries that added the most people since 1970 are African — led by Nigeria'
          '<br><sup>People added 1970–2022, in millions · <span style="color:#0072B2">blue = African countries</span></sup>',
)
fig.update_traces(texttemplate='%{x:,.0f}M', textposition='outside', cliponaxis=False)
fig.update_layout(showlegend=False, height=520, margin=dict(r=40))
fig.update_xaxes(visible=False, range=[0, 1000])
fig.update_yaxes(title=None, categoryorder='array', categoryarray=top['Country'].tolist())
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** **Scatter plot (log–log).** Two numeric variables and a question about their relationship call for a scatter plot, with continent as colour and a fitted trend line so the pattern and the exceptions are visible. Both axes are logarithmic because area and population span about six orders of magnitude, and on a linear scale nearly all countries would be squashed into a corner. A palette from the CVD-safe Okabe–Ito set separates continents; only the six countries furthest from the trend are annotated.

In [16]:
# Q3
import plotly.express as px

COLORS = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
          'North America': '#56B4E9', 'Oceania': '#999999', 'South America': '#009E73'}

# trend line in log-log space and each country's distance from it
b, a = np.polyfit(np.log10(df['Area']), np.log10(df['2022 Population']), 1)
df3 = df.assign(Resid=np.log10(df['2022 Population']) - (a + b * np.log10(df['Area'])))
outliers = pd.concat([df3.nlargest(3, 'Resid'), df3.nsmallest(3, 'Resid')])
r = np.corrcoef(np.log10(df['Area']), np.log10(df['2022 Population']))[0, 1]

xs = np.logspace(np.log10(df['Area'].min()), np.log10(df['Area'].max()), 50)
fit = pd.DataFrame({'Area': xs, 'Population': 10 ** (a + b * np.log10(xs))})

fig = px.scatter(
    df3, x='Area', y='2022 Population', color='Continent', hover_name='Country', log_x=True, log_y=True,
    color_discrete_map=COLORS, opacity=0.8, template='plotly_white',
    title='Bigger countries tend to have more people — but city-states and Greenland break the pattern'
          f'<br><sup>Land area vs. 2022 population, log scales · log–log correlation r = {r:.2f}</sup>',
)
fig.update_traces(marker_size=8)
fig.add_traces(px.line(fit, x='Area', y='Population')
                 .update_traces(line=dict(color='#444', dash='dash', width=1.5), hoverinfo='skip', showlegend=False).data)

for _, row in outliers.iterrows():
    fig.add_annotation(x=np.log10(row['Area']), y=np.log10(row['2022 Population']), text=row['Country'],
                       showarrow=True, arrowhead=0, arrowcolor='#999', ax=35, ay=-25 if row['Resid'] > 0 else 25,
                       font=dict(size=11, color='#333'))

fig.update_layout(height=560, legend=dict(orientation='h', y=1.02, yanchor='bottom', x=0, title=None),
                  margin=dict(t=110))
fig.update_xaxes(title='Land area (km², log scale)', showgrid=False)
fig.update_yaxes(title='Population 2022 (log scale)', gridcolor='#EEE')
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** **Box plot (log y-axis).** The question asks for spread within and across groups, which is what a box plot is for: median, quartiles and outliers per continent, side by side. A violin would be less reliable here because several continents have few countries (Oceania, South America). Density is extremely right-skewed (2 to 23,000 people/km²), so the axis is logarithmic. Continents are sorted by median; the highest-median continent gets the accent colour and the three densest places are labelled.

In [17]:
# Q4
import plotly.express as px

GREY, ACCENT = '#B8B8B8', '#0072B2'

order = df.groupby('Continent')['Density'].median().sort_values(ascending=False).index.tolist()
df4 = df.assign(Highlight=np.where(df['Continent'] == order[0], 'top', 'other'))

fig = px.box(
    df4, x='Continent', y='Density', color='Highlight', points='outliers', hover_name='Country', log_y=True,
    category_orders={'Continent': order}, color_discrete_map={'top': ACCENT, 'other': GREY},
    template='plotly_white',
    title='Countries within a continent differ in density far more than continents differ from each other'
          '<br><sup>Population density 2022, people per km² (log scale) · line = median, box = middle 50% of countries</sup>',
)
fig.update_layout(boxmode='overlay', showlegend=False, height=520)

for _, row in df.nlargest(3, 'Density').iterrows():
    fig.add_annotation(x=row['Continent'], y=np.log10(row['Density']), text=row['Country'], showarrow=True,
                       arrowhead=0, arrowcolor='#999', ax=45, ay=0, font=dict(size=11, color='#333'))

fig.update_xaxes(title=None, showgrid=False)
fig.update_yaxes(title=None, gridcolor='#EEE')
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** **Choropleth map.** The question is explicitly about *where* in the world, so a map is the right chart. Colour is a diverging orange–blue scale (CVD-safe, no red/green) with white fixed at 0%, so orange means shrinking and blue means growing. The scale is capped at +160% because a few small outliers (Qatar +317%) would otherwise wash out the rest; hover still shows the true value. Small island states are hard to see on a map, a known limitation.

In [18]:
# Q5
import plotly.express as px

df5 = df.assign(Change=(df['2022 Population'] / df['2000 Population'] - 1) * 100)
n_shrank = int((df5['Change'] < 0).sum())
n_europe = int(((df5['Change'] < 0) & (df5['Continent'] == 'Europe')).sum())

lo, hi = -40, 160
zero = (0 - lo) / (hi - lo)          # position of 0% on the colour scale, so white = no change

fig = px.choropleth(
    df5, locations='CCA3', locationmode='ISO-3', color='Change', hover_name='Country',
    color_continuous_scale=[[0, '#D55E00'], [zero, '#F5F5F5'], [1, '#0072B2']], range_color=[lo, hi],
    template='plotly_white',
    title=f'{n_shrank} countries shrank between 2000 and 2022 — {n_europe} of them in Europe — while every African country grew'
          '<br><sup>Population change 2000–2022, % · <span style="color:#D55E00">orange = shrank</span>, '
          '<span style="color:#0072B2">blue = grew</span> (scale capped at +160%)</sup>',
)
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth', bgcolor='rgba(0,0,0,0)')
fig.update_layout(height=560, margin=dict(l=0, r=0, b=0),
                  coloraxis_colorbar=dict(title='% change', thickness=12, len=0.6))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** **Treemap.** The data is hierarchical (continent → country) and the question is about parts of a whole at both levels, which a treemap shows in one figure: tile area is population, nesting shows the continent. Colour encodes continent (Okabe–Ito, CVD-safe, same colours as Q3), labels show each tile's share of the world, and text on tiles that are too small is hidden automatically.

In [19]:
# Q6
import plotly.express as px

COLORS = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
          'North America': '#56B4E9', 'Oceania': '#999999', 'South America': '#009E73', '(?)': 'white'}

fig = px.treemap(
    df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
    color='Continent', color_discrete_map=COLORS, template='plotly_white',
    title="Asia holds 59% of the world's 8 billion people — China and India alone are over a third"
          '<br><sup>2022 population · tile area = population, labels = share of world</sup>',
)
fig.update_traces(texttemplate='%{label}<br>%{percentRoot:.1%}', root_color='white',
                  marker_line_width=1, marker_line_color='white', hovertemplate='%{label}<br>%{value:,.0f} people<extra></extra>')
fig.update_layout(uniformtext=dict(minsize=9, mode='hide'), height=620, margin=dict(l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** **100% stacked bar chart.** Shares of a whole that always sum to 100%, tracked across several years: a 100% stacked bar shows composition and its change together. The two groups that move get colour (Africa blue = rising, Europe orange = falling), Asia and the rest are grey. Segments are labelled with their %, and the group names sit next to the last bar instead of a legend. Years are shown as categories, so the uneven gaps (e.g. 2020 → 2022) do not distort the picture.

In [20]:
# Q7
import plotly.express as px

COLORS = {'Africa': '#0072B2', 'Asia': '#9A9A9A', 'Europe': '#E69F00', 'Rest of the world': '#D9D9D9'}
order = ['Africa', 'Asia', 'Europe', 'Rest of the world']          # bottom -> top

d7 = df_long.assign(Group=np.where(df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
                                   df_long['Continent'], 'Rest of the world'))
d7 = d7.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
d7['Share'] = d7['Population'] / d7.groupby('Year')['Population'].transform('sum') * 100

fig = px.bar(
    d7, x='Year', y='Share', color='Group', text='Share', template='plotly_white',
    category_orders={'Group': order}, color_discrete_map=COLORS,
    title="Africa's share of world population rose from 10% to 18% since 1970, while Europe's fell from 18% to 9%"
          '<br><sup>Share of world population by region, %</sup>',
)
fig.update_traces(texttemplate='%{y:.0f}%', textposition='inside', insidetextanchor='middle')
fig.for_each_trace(lambda t: t.update(textfont_color='white' if t.name == 'Africa' else '#222'))

# direct labels next to the last bar (replace the legend)
last = d7[d7['Year'] == d7['Year'].max()].set_index('Group')['Share'].reindex(order)
bottom = last.cumsum() - last
for g in order:
    fig.add_annotation(xref='paper', x=1.0, xshift=10, xanchor='left', y=bottom[g] + last[g] / 2,
                       text=f'<b>{g}</b>', showarrow=False,
                       font=dict(color=COLORS[g] if g in ('Africa', 'Europe') else '#666'))

fig.update_layout(showlegend=False, height=520, margin=dict(r=150))
fig.update_xaxes(type='category', title=None)
fig.update_yaxes(title=None, showgrid=False, showticklabels=False)
fig.show()